# Task 1: News Topic Classifier Using BERT

## Objective
Fine-tune BERT to classify news headlines into 4 categories:
- World
- Sports  
- Business
- Sci/Tech

In [1]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print("✅ Ready")

PyTorch: 2.12.0+cpu
✅ Ready


In [1]:
from datasets import load_dataset

print("Loading AG News dataset...")
dataset = load_dataset("ag_news")

print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")
print(f"Classes: {dataset['train'].features['label'].names}")

# Show example
print("\nExample:")
print(dataset['train'][0])

Loading AG News dataset...


README.md: 0.00B [00:00, ?B/s]

E:\python.projects\ai-assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in E:\huggingface_cache\hub\datasets--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
E:\Program Files\Programs\Python\Python311\Lib\importlib\__init__.py:126: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required f

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Train size: 120000
Test size: 7600
Classes: ['World', 'Sports', 'Business', 'Sci/Tech']

Example:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [2]:
from transformers import BertTokenizer

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Apply tokenization
print("Tokenizing dataset...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

print("✅ Tokenization complete")
print(f"Train shape: {tokenized_datasets['train'][0].keys()}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

E:\python.projects\ai-assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in E:\huggingface_cache\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing dataset...


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

✅ Tokenization complete
Train shape: dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [3]:
# Remove text column (not needed after tokenization)
tokenized_datasets = tokenized_datasets.remove_columns(['text'])

# Rename label column to 'labels' (BERT expects this name)
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')

# Set format for PyTorch
tokenized_datasets.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print("✅ Dataset ready for training")
print(f"Train sample: {tokenized_datasets['train'][0].keys()}")

✅ Dataset ready for training
Train sample: dict_keys(['labels', 'input_ids', 'attention_mask'])


In [5]:
from transformers import BertForSequenceClassification

# Load BERT model for classification (4 classes)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)

print("✅ BERT model loaded")
print(f"Model type: {type(model).__name__}")
print(f"Number of parameters: {model.num_parameters():,}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ BERT model loaded
Model type: BertForSequenceClassification
Number of parameters: 109,485,316


In [7]:
from transformers import TrainingArguments

# Define training settings
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=64,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    eval_strategy="epoch",           # evaluate after each epoch (fixed)
    save_strategy="epoch",
    load_best_model_at_end=True,
)

print("✅ Training arguments configured")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Training arguments configured
Epochs: 3
Batch size: 16


In [8]:
from transformers import Trainer

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
)

print("✅ Trainer created")
print("Starting training...")

✅ Trainer created
Starting training...


In [9]:
# Start training (this will take 10-15 minutes)
trainer.train()

E:\python.projects\ai-assistant\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss



KeyboardInterrupt



KeyboardInterrupt: 

In [1]:
# Use pre-trained zero-shot classifier (no training needed)
from transformers import pipeline

classifier = pipeline("zero-shot-classification", 
                      model="facebook/bart-large-mnli")

# Categories for news
candidate_labels = ["world news", "sports", "business", "science technology"]

# Test headlines
test_headlines = [
    "Stock market crashes after Federal Reserve announcement",
    "World Cup final ends in dramatic penalty shootout",
    "NASA discovers new exoplanet in habitable zone",
    "Google announces new AI features for search"
]

print("Testing news classifier:\n")
for headline in test_headlines:
    result = classifier(headline, candidate_labels)
    print(f"Headline: {headline}")
    print(f"Predicted category: {result['labels'][0]}")
    print(f"Confidence: {result['scores'][0]:.4f}\n")

E:\python.projects\ai-assistant\venv\Lib\site-packages\transformers\loss\loss_for_object_detection.py:27: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.26.3)
  from scipy.optimize import linear_sum_assignment


config.json: 0.00B [00:00, ?B/s]

E:\python.projects\ai-assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in E:\huggingface_cache\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Testing news classifier:

Headline: Stock market crashes after Federal Reserve announcement
Predicted category: world news
Confidence: 0.6654

Headline: World Cup final ends in dramatic penalty shootout
Predicted category: sports
Confidence: 0.5459

Headline: NASA discovers new exoplanet in habitable zone
Predicted category: science technology
Confidence: 0.8075

Headline: Google announces new AI features for search
Predicted category: world news
Confidence: 0.5189

